In [4]:
# ============================================================
# PHASE 4 - ETL: Load Star Schema into MySQL
# ============================================================

import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load credentials from .env
load_dotenv(r"C:\Users\yipch\ecommerce-analytics-portfolio\.env")

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASSWORD", "")
DB_NAME = os.getenv("DB_NAME", "ecommerce_analytics")

# Build connection string
conn_str = f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine   = create_engine(conn_str, echo=False)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT VERSION()"))
    version = result.fetchone()[0]
    print(f"✅ Connected to MySQL")
    print(f"   Version  : {version}")
    print(f"   Database : {DB_NAME}")

EXPORT_DIR = r"C:\Users\yipch\ecommerce-analytics-portfolio\data\exports"
print(f"\n✅ Setup complete. Ready to load tables.")

✅ Connected to MySQL
   Version  : 9.6.0
   Database : ecommerce_analytics

✅ Setup complete. Ready to load tables.


In [7]:
# ============================================================
# STEP 5 - Load all dimension tables
# Rule: dims must load BEFORE fact (foreign key order)
# ============================================================

def load_table(filename, table_name, engine,
               chunksize=10_000, dtype_map=None):
    """Load a CSV into MySQL in chunks — memory safe."""

    filepath = os.path.join(EXPORT_DIR, filename)
    df = pd.read_csv(filepath)

    print(f"\n⏳ Loading {table_name}...")
    print(f"   Rows to insert : {df.shape[0]:,}")

    # Replace NaN with None (MySQL-compatible nulls)
    df = df.where(pd.notnull(df), None)

    df.to_sql(
        name        = table_name,
        con         = engine,
        if_exists   = "append",   # table already created in Step 2
        index       = False,
        chunksize   = chunksize,
        method      = "multi",
    )

    # Verify row count in MySQL
    with engine.connect() as conn:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}"))
        db_count = result.fetchone()[0]

    match = "✅" if db_count == df.shape[0] else "⚠️ MISMATCH"
    print(f"   {match} Inserted: {db_count:,} rows in MySQL")
    return df.shape[0], db_count

# ─────────────────────────────────────────
# Load dimensions (order matters!)
# ─────────────────────────────────────────
print("=" * 55)
print("  LOADING DIMENSION TABLES")
print("=" * 55)

load_table("dim_date.csv",     "dim_date",     engine)
load_table("dim_products.csv", "dim_products", engine)
load_table("dim_users.csv",    "dim_users",    engine)
load_table("dim_sessions.csv", "dim_sessions", engine)

print("\n✅ All dimension tables loaded!")

  LOADING DIMENSION TABLES

⏳ Loading dim_date...
   Rows to insert : 1,288,421
   ✅ Inserted: 1,288,421 rows in MySQL

⏳ Loading dim_products...
   Rows to insert : 95,684
   ✅ Inserted: 95,684 rows in MySQL

⏳ Loading dim_users...
   Rows to insert : 99,658
   ✅ Inserted: 99,658 rows in MySQL

⏳ Loading dim_sessions...
   Rows to insert : 341,526
   ✅ Inserted: 341,526 rows in MySQL

✅ All dimension tables loaded!


In [8]:
# ============================================================
# STEP 6 - Load fact_events
# Largest table — loaded in chunks with progress tracking
# ============================================================

from tqdm import tqdm

filepath = os.path.join(EXPORT_DIR, "fact_events.csv")
df_fact  = pd.read_csv(filepath)
df_fact  = df_fact.where(pd.notnull(df_fact), None)

print(f"⏳ Loading fact_events...")
print(f"   Total rows : {df_fact.shape[0]:,}")

CHUNK = 10_000
total_chunks = (len(df_fact) // CHUNK) + 1

for i, start in enumerate(tqdm(range(0, len(df_fact), CHUNK),
                                desc="Inserting chunks",
                                total=total_chunks)):
    chunk = df_fact.iloc[start : start + CHUNK]
    chunk.to_sql(
        name      = "fact_events",
        con       = engine,
        if_exists = "append",
        index     = False,
        method    = "multi",
    )

# Verify
with engine.connect() as conn:
    result   = conn.execute(text("SELECT COUNT(*) FROM fact_events"))
    db_count = result.fetchone()[0]

match = "✅" if db_count == df_fact.shape[0] else "⚠️ MISMATCH"
print(f"\n{match} fact_events in MySQL : {db_count:,} rows")

⏳ Loading fact_events...
   Total rows : 1,605,102


Inserting chunks: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 161/161 [06:23<00:00,  2.38s/it]



✅ fact_events in MySQL : 1,605,102 rows


In [9]:
# ============================================================
# STEP 7 - Full verification: row counts + sample queries
# ============================================================

print("=" * 55)
print("  MYSQL LOAD VERIFICATION")
print("=" * 55)

tables = ["dim_date", "dim_products", "dim_users",
          "dim_sessions", "fact_events"]

print(f"\n{'Table':<20} {'MySQL Rows':>12} {'CSV Rows':>12} {'Match':>8}")
print("-" * 55)

with engine.connect() as conn:
    for table in tables:
        # MySQL count
        r        = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        db_rows  = r.fetchone()[0]

        # CSV count
        csv_path = os.path.join(EXPORT_DIR, f"{table}.csv")
        csv_rows = sum(1 for _ in open(csv_path)) - 1  # minus header

        match = "✅" if db_rows == csv_rows else "⚠️"
        print(f"{table:<20} {db_rows:>12,} {csv_rows:>12,} {match:>8}")

print("\n" + "=" * 55)
print("  BUSINESS SANITY QUERIES")
print("=" * 55)

sanity_queries = {
    "Total revenue (purchases only)":
        "SELECT ROUND(SUM(revenue),2) AS total_revenue FROM fact_events",

    "Events by type":
        "SELECT event_type, COUNT(*) AS cnt FROM fact_events GROUP BY event_type ORDER BY cnt DESC",

    "Top 5 brands by revenue":
        """SELECT p.brand,
                  ROUND(SUM(f.revenue),2) AS revenue,
                  COUNT(*) AS purchases
           FROM fact_events f
           JOIN dim_products p ON f.product_id = p.product_id
           WHERE f.event_type = 'purchase'
           GROUP BY p.brand
           ORDER BY revenue DESC
           LIMIT 5""",

    "Revenue by month":
        """SELECT month,
                  ROUND(SUM(revenue),2) AS revenue,
                  COUNT(*) AS purchases
           FROM fact_events
           WHERE event_type = 'purchase'
           GROUP BY month
           ORDER BY month""",
}

with engine.connect() as conn:
    for label, query in sanity_queries.items():
        print(f"\n📊 {label}:")
        result = conn.execute(text(query))
        rows   = result.fetchall()
        cols   = result.keys()
        df_out = pd.DataFrame(rows, columns=cols)
        print(df_out.to_string(index=False))

  MYSQL LOAD VERIFICATION

Table                  MySQL Rows     CSV Rows    Match
-------------------------------------------------------
dim_date                1,288,421    1,288,421        ✅
dim_products               95,684       95,684        ✅
dim_users                  99,658       99,658        ✅
dim_sessions              341,526      341,526        ✅
fact_events             1,605,102    1,605,102        ✅

  BUSINESS SANITY QUERIES

📊 Total revenue (purchases only):
total_revenue
   7416954.89

📊 Events by type:
event_type     cnt
      view 1525176
      cart   55324
  purchase   24602

📊 Top 5 brands by revenue:
  brand    revenue  purchases
  apple 3452603.80       4472
samsung 1551972.69       5612
 xiaomi  315076.42       1932
unknown  301554.85       1955
 huawei  128228.49        615

📊 Revenue by month:
   month    revenue  purchases
2019-Nov 3856928.12      12909
2019-Oct 3560026.77      11693


In [10]:
# ============================================================
# STEP 8 - Save reusable SQL files to /sql folder
# These become your portfolio SQL samples
# ============================================================

SQL_DIR = r"C:\Users\yipch\ecommerce-analytics-portfolio\sql"

sql_files = {

"01_schema_check.sql": """
-- Schema verification queries
USE ecommerce_analytics;
SHOW TABLES;
SELECT TABLE_NAME, TABLE_ROWS
FROM information_schema.TABLES
WHERE TABLE_SCHEMA = 'ecommerce_analytics';
""",

"02_revenue_by_month.sql": """
-- Monthly revenue trend
USE ecommerce_analytics;
SELECT
    month,
    COUNT(*)                    AS total_purchases,
    ROUND(SUM(revenue), 2)      AS total_revenue,
    ROUND(AVG(price), 2)        AS avg_order_value
FROM fact_events
WHERE event_type = 'purchase'
GROUP BY month
ORDER BY month;
""",

"03_top_brands.sql": """
-- Top 10 brands by revenue
USE ecommerce_analytics;
SELECT
    p.brand,
    COUNT(*)                    AS purchases,
    ROUND(SUM(f.revenue), 2)    AS revenue,
    ROUND(AVG(f.price), 2)      AS avg_price
FROM fact_events f
JOIN dim_products p ON f.product_id = p.product_id
WHERE f.event_type = 'purchase'
  AND p.brand != 'unknown'
GROUP BY p.brand
ORDER BY revenue DESC
LIMIT 10;
""",

"04_conversion_funnel.sql": """
-- Overall conversion funnel
USE ecommerce_analytics;
SELECT
    event_type,
    COUNT(*)                                        AS events,
    COUNT(DISTINCT user_id)                         AS unique_users,
    ROUND(COUNT(*) * 100.0 /
          SUM(COUNT(*)) OVER (), 2)                 AS pct_of_total
FROM fact_events
GROUP BY event_type
ORDER BY FIELD(event_type, 'view','cart','purchase');
""",

"05_hourly_pattern.sql": """
-- Hourly traffic + purchase pattern
USE ecommerce_analytics;
SELECT
    d.hour,
    COUNT(*)                                    AS total_events,
    SUM(CASE WHEN f.event_type='purchase'
             THEN 1 ELSE 0 END)                 AS purchases,
    ROUND(SUM(f.revenue), 2)                    AS revenue
FROM fact_events f
JOIN dim_date d ON f.date_id = d.date_id
GROUP BY d.hour
ORDER BY d.hour;
"""
}

for filename, content in sql_files.items():
    path = os.path.join(SQL_DIR, filename)
    with open(path, "w") as f:
        f.write(content.strip())
    print(f"✅ Saved: sql/{filename}")

print("\n🎯 Open these in DBeaver/Workbench to run and explore!")

✅ Saved: sql/01_schema_check.sql
✅ Saved: sql/02_revenue_by_month.sql
✅ Saved: sql/03_top_brands.sql
✅ Saved: sql/04_conversion_funnel.sql
✅ Saved: sql/05_hourly_pattern.sql

🎯 Open these in DBeaver/Workbench to run and explore!
